# Ground Truth : le pipeline complet sur Colab (GPU T4)

1. Exécution > Modifier le type d'exécution > **GPU T4**.
2. Exécution > **Tout exécuter**. Compter environ 2 h la première fois : téléchargement des modèles, puis l'OCR des pages numérisées, qui est l'étape la plus longue.

Le notebook suit exactement le contrat de la finale (`setup.sh`, puis `run.sh index`, `update`, `answer`), mesure la durée de chaque étape et note les réponses avec le générateur et le juge officiels.

Si Colab demande l'accès au secret HF_TOKEN, tu peux cliquer sur Annuler.

In [ ]:
%cd /content
!test -d Ground-Truth || git clone -q https://github.com/blaster28/Ground-Truth.git
!git -C Ground-Truth pull -q
%cd /content/Ground-Truth
!git log --oneline -1
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader

## Installation : dépendances et poids des modèles (seule étape avec réseau)

In [ ]:
%%time
!bash setup.sh
!python corpus/download_corpus.py

## Indexation du corpus v1 (limite : 2 h)

Les lignes `[ocr]` montrent l'avancement. Le résultat de l'OCR est gardé dans `ocr_cache.json` : si la session est coupée, relancer la cellule reprend là où elle s'était arrêtée.

In [ ]:
%%time
!bash run.sh index corpus/pdf /content/index

## Validation v1 : réponses avec le générateur officiel, note du juge officiel

In [ ]:
%%time
!bash run.sh answer /content/index data/validation_v1.json runs/val1.json

In [ ]:
!python -m groundtruth.evaluate --pred runs/val1.json --gold data/validation_v1.json --judge hf --out runs/rapport_val1.json

## Durée par question (limite : 30 s en moyenne, recherche comprise)

In [ ]:
import json
log = [json.loads(l) for l in open('runs/generator_log.jsonl') if l.strip()]
log = [l for l in log if l['backend'] == 'hf']
print(f"{len(log)} appels, {sum(l['seconds'] for l in log) / max(len(log), 1):.1f} s par appel en moyenne, "
      f"{max(l['n_in'] for l in log)} tokens d'entrée au maximum")

## Leaderboard v1 : les 200 questions de classement

In [ ]:
%%time
!bash run.sh answer /content/index data/classement_v1_questions.json runs/classement_v1.json

## Palier 3 : mise à jour avec les 20 documents v2 (limite : 20 min), puis questions v2

In [ ]:
%%time
!bash run.sh update /content/index corpus/pdf_v2

In [ ]:
%%time
!bash run.sh answer /content/index data/validation_v2.json runs/val2.json
!python -m groundtruth.evaluate --pred runs/val2.json --gold data/validation_v2.json --judge hf --out runs/rapport_val2.json

In [ ]:
%%time
!bash run.sh answer /content/index data/classement_v2_questions.json runs/classement_v2.json

## Récupérer les résultats

Télécharge `resultats.zip` (réponses, rapports, journal du générateur, cache OCR) et décompresse-le à la racine du dépôt sur ta machine.

In [ ]:
!cp /content/index/ocr_cache.json runs/ocr_cache_t4.json
!zip -q -r /content/resultats.zip runs
from google.colab import files
files.download('/content/resultats.zip')